# 03 — Leverage, margin and account measures

[Module 02](../course/02-trading-terminology.md) · [Notebook index](README.md)

**Purpose:** Understand how much margin a position may require, and how open losses affect account equity. All numbers are hypothetical. Run cells in order.

For simple Forex margin based on base-currency notional, constant leverage and no extra margin multipliers:

$$\text{Notional in account currency}=\text{Lots}\times\text{Contract size}\times\text{Base-to-account rate}$$

$$\text{Required margin}=\frac{\text{Notional in account currency}}{\text{Leverage ratio}}$$

1:100 leverage means the ratio is 100. Base-to-account rate is account-currency units per one base-currency unit; use 1 if they match. This is an estimate, not a universal margin formula.

Simplified account measures (no credits, blocked funds or separate unbooked charges):
 
$$\text{Equity}=\text{Balance}+\text{Floating P/L}$$

$$\text{Free margin}=\text{Equity}-\text{Used margin}$$

$$\text{Margin level (percent)}=100\times\frac{\text{Equity}}{\text{Used margin}}$$

With no used margin, the ratio is undefined; the notebook returns None. Broker thresholds for margin calls and stop-outs differ. Margin is collateral, not the maximum possible loss.


In [ ]:
from math import isfinite, isclose

def positive(value, name):
    if not isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be finite and positive")
    return value

def finite(value, name):
    if not isfinite(value):
        raise ValueError(f"{name} must be finite")
    return value

def expect_value_error(function, *args):
    try:
        function(*args)
    except ValueError:
        return
    raise AssertionError("Expected ValueError")

def estimate_forex_margin(lots, contract_size, leverage_ratio, base_to_account=1.0):
    notional = (positive(lots, "lots") * positive(contract_size, "contract size")
                * positive(base_to_account, "conversion rate"))
    return notional / positive(leverage_ratio, "leverage ratio")

def account_measures(balance, floating_pnl, used_margin):
    finite(balance, "balance")
    finite(floating_pnl, "floating P/L")
    if not isfinite(used_margin) or used_margin < 0:
        raise ValueError("used margin must be finite and nonnegative")
    equity = balance + floating_pnl
    return {
        "equity": equity,
        "free_margin": equity - used_margin,
        "margin_level_percent": equity / used_margin * 100 if used_margin else None,
    }


## Worked example

A USD account opens 0.10 lots EUR/USD with a contract size of 100,000 EUR, base-to-account rate 1.10 USD per EUR and 1:100 leverage. The USD notional is $11,000 and estimated margin is $110.

Separately, take a USD account with $1,000 balance, −$100 floating P/L and $110 total used margin. Equity is $900, free margin is $790 and margin level is approximately 818.18%.


In [ ]:
margin = estimate_forex_margin(0.10, 100_000, 100, 1.10)
print(f"Estimated margin: USD {margin:.2f}")
account = account_measures(1000, -100, margin)
print(f"Equity: USD {account['equity']:.2f}")
print(f"Free margin: USD {account['free_margin']:.2f}")
print(f"Margin level: {account['margin_level_percent']:.2f}%")


## Try it yourself

Keep the same position but change leverage to 1:50. Estimated margin becomes $220. With equity of $900 and total used margin of $220, free margin is $680 and margin level is approximately 409.09%. For unchanged position size, a price movement produces the same gross P/L regardless of leverage; higher leverage makes larger exposure possible, not automatically safer.


In [ ]:
assert isclose(estimate_forex_margin(0.10, 100_000, 100, 1.10), 110)
assert isclose(estimate_forex_margin(0.10, 100_000, 50, 1.10), 220)
assert isclose(account_measures(1000, -100, 110)["margin_level_percent"], 900 / 110 * 100)
assert account_measures(1000, -100, 220)["free_margin"] == 680
assert account_measures(1000, 0, 0)["margin_level_percent"] is None
assert account_measures(100, -90, 20)["free_margin"] == -10
expect_value_error(estimate_forex_margin, 0.1, 100_000, 0)
expect_value_error(account_measures, 1000, 0, -1)
print("All margin and account checks passed.")


## Online calculator and limitations

- [Myfxbook margin calculator](https://www.myfxbook.com/forex-calculators/margin-calculator)
- [MT5 account measures and trade execution](https://www.metatrader5.com/en/terminal/help/trading/performing_deals)
- [MQL5 OrderCalcMargin](https://www.mql5.com/en/docs/trading/ordercalcmargin)

Online calculators may use live rates. Actual margin depends on symbol calculation mode, broker leverage tiers, margin rates, hedging/netting, pending orders and existing positions. Real equity can include credits, commissions and blocked funds; use the platform's actual account values. OrderCalcMargin estimates a planned order in isolation, not incremental account margin in every case. Never treat margin as a stop loss or a maximum risk amount.
